In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.

# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM

!pip install chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------
# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader("/content/ai_research.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------
# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------
# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

FileNotFoundError: [Errno 2] No such file or directory: '/content/ai_research.pdf'

In [ ]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.




In [ ]:

print("Loading legal document...")

pdf_path = "/content/data_privacy_regulation.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text()

print("Document loaded successfully")
print("Total characters:", len(text))


print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])



print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Loading legal document...
Document loaded successfully
Total characters: 3427

Splitting document into chunks...
Total Chunks Created: 8

Example Chunk:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing, 

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: Why to do things ?


Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
ke data-driven decisions.
With the rapid growth of data, the demand for skilled data scientists continues to rise.
Understanding the basics of data science, machine learning, and data analysis is an essential skill
for students and professionals in the modern digital economy.
 Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing,  ilding models that learn patterns from data.

Data Visualization

Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
achine learning library.

TensorFlow / PyTorch – Deep learning frameworks.
5. Applications of Data Science

Healthcare – Disease prediction and medical imaging.

Finance – Fraud detection and risk analysis.

E-commerce – Recommendation systems.

Transportation – Traffic prediction and route optimization.

Social Media – Sentiment analysis and trend detection.
Conclusion
Data Science is transforming industries by enabling organizations to make data-driven decisions.
With the rapid growth of  Machine Learning in Data Science
Machine learning is one of the most important components of data science. It allows computers to
automatically learn patterns from data and make predictions without being explicitly programmed.
Types of Machine Learning:

Supervised Learning – Models are trained using labeled data where the correct output i

In [ ]:
# Scenario: University Library Assistant
# A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.
# How It Works
# - Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
# - Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
# - Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
# - Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
# - LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
# - Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.


In [ ]:


print("University Library Assistant...")

pdf_path = "/content/Introduction_to_Data_Science.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text()

print("Document loaded successfully")
print("Total characters:", len(text))


print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])



print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

University Library Assistant...
Document loaded successfully
Total characters: 3427

Splitting document into chunks...
Total Chunks Created: 8

Example Chunk:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing, 

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Old collection deleted
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: what is not dataScience?


Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing,  Science involves collecting, cleaning, analyzing, and interpreting large datasets. It uses
techniques from statistics, machine learning, and data visualization to identify patterns and trends
within data.
Key Components of Data Science:

Data Collection – Gathering raw data from different sources.

Data Cleaning – Removing noise, errors, o

In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fo

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0b91f3b9b7fd13fda.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Scenario: Healthcare Clinical Research Assistant
Context

Hospitals and medical research teams regularly analyze large medical documents such as:

clinical trial reports

treatment guidelines

drug safety reports

patient care protocols

These documents are very long and technical, often containing:

medical terminology

treatment procedures

dosage guidelines

adverse drug reactions

Doctors and healthcare researchers often spend hours searching for specific information, such as:

recommended drug dosages

treatment protocols for specific diseases

safety warnings and contraindications

In [ ]:
# ==============================================================
# HEALTHCARE CLINICAL RESEARCH ASSISTANT
# RAG + GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("medical_docs")
except:
    pass

collection = client.create_collection("medical_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Chunk Text
# --------------------------------------------------------------

def chunk_text(text, chunk_size=600, overlap=80):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded Medical PDF
# --------------------------------------------------------------

def process_pdf(file):

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    embeddings = embedding_model.encode(chunks).tolist()

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(chunks))]
    )

    return f"Medical document processed successfully. {len(chunks)} sections stored."


# --------------------------------------------------------------
# STEP 8 — Retrieve Relevant Medical Text
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    return docs


# --------------------------------------------------------------
# STEP 9 — Generate Answer
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a clinical research assistant helping doctors and medical researchers.

Use ONLY the medical context below to answer the question clearly.

Context:
{context}

Question:
{query}

Provide a clear medical explanation.
If the answer is not found, say "Not mentioned in document".
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    return response[0]["generated_text"]


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a medical question."

    answer = answer_question(question)

    return answer


# --------------------------------------------------------------
# STEP 11 — Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Healthcare Clinical Research Assistant")

    gr.Markdown("""
Upload medical documents such as:

• Clinical trial reports
• Treatment guidelines
• Drug safety reports
• Patient care protocols

Then ask questions like:

• What dosage is recommended for this drug?
• What are the adverse reactions mentioned?
• What treatment protocol is recommended?
• Are there any contraindications?
""")

    pdf_input = gr.File(label="Upload Medical PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(label="Ask a Medical Question")

    answer_box = gr.Textbox(label="Answer", lines=12)

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch App
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4f3fe6119be34889fb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Scenario: "The Environmental Policy Compliance Assistant"
Background
You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
The company must ensure compliance to avoid fines and reputational damage.

Challenge
The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
Your task is to:
- Process the regulation document so the assistant can store and understand it.
- Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
- Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

Roles
- Learner (You): Environmental compliance officer using the assistant.
- Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
- Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

🔄 Flow of the Scenario
- Upload Environmental Regulation PDF
Example: “National Carbon Emissions Act 2026”.
- System Processes Document
- Splits into chunks.
- Embeds into vector database.
- Stores for retrieval.
- Ask Questions
- “What is the maximum carbon emission allowed per factory per year?”
- “What penalties apply if hazardous waste is not disposed of properly?”
- “What renewable energy targets must we meet by 2030?”
- Assistant Responds
- Retrieves relevant chunks.
- Generates compliance-focused answers.
- Provides short, clear guidance.
- Outcome
- Learners practice extracting environmental obligations.
- Managers receive summarized compliance insights.
- Executives gain confidence in sustainability strategy alignment.

🎯 Training Objective
This scenario helps learners:
- Understand how RAG systems can support environmental compliance.
- Practice formulating precise queries to extract obligations.
- Experience how AI can simplify complex environmental regulations into actionable steps.

👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier) for this environmental context, so you can upload it into your assistant and simulate queries?

In [ ]:
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ------------------------------------------------
# STEP 1 — Initialize Models
# ------------------------------------------------

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

client = chromadb.Client()
collection = None


# ------------------------------------------------
# STEP 2 — Chunking Function
# ------------------------------------------------

def chunk_text(text, chunk_size=800, overlap=100):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# ------------------------------------------------
# STEP 3 — Process Uploaded PDF
# ------------------------------------------------

def process_pdf(file):

    global collection

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text

    chunks = chunk_text(text)

    try:
        client.delete_collection("environment_collection")
    except:
        pass

    collection = client.create_collection("environment_collection")

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return "✅ Document processed successfully! You can now ask compliance questions."


# ------------------------------------------------
# STEP 4 — Retriever
# ------------------------------------------------

def retrieve(query, k=4):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ------------------------------------------------
# STEP 5 — Question Answering
# ------------------------------------------------

def ask_question(question):

    if collection is None:
        return "⚠️ Please upload a regulation PDF first."

    context_docs = retrieve(question)

    context = " ".join(context_docs)

    prompt = f"""
You are an Environmental Policy Compliance Assistant.

Use ONLY the regulation context below to answer the question.
Provide short and clear compliance guidance for engineers, sustainability officers, and executives.

If the answer is not present in the regulation, say:
"Not found in regulation."

Context:
{context}

Question:
{question}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.3
    )

    return response[0]["generated_text"]


# ------------------------------------------------
# STEP 6 — Gradio Interface
# ------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🌍 Environmental Policy Compliance Assistant")

    gr.Markdown(
        "Upload an environmental regulation PDF and ask compliance questions about emissions, waste management, and renewable energy targets."
    )

    file_input = gr.File(label="Upload Environmental Regulation PDF")

    upload_button = gr.Button("Process Document")

    status_output = gr.Textbox(label="System Status")

    question_input = gr.Textbox(
        label="Ask a Compliance Question",
        placeholder="Example: What is the maximum carbon emission allowed per factory?"
    )

    answer_output = gr.Textbox(label="Compliance Guidance")

    ask_button = gr.Button("Ask Question")

    upload_button.click(process_pdf, inputs=file_input, outputs=status_output)

    ask_button.click(ask_question, inputs=question_input, outputs=answer_output)


demo.launch()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://53b3619882946cb039.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
